<a href="https://colab.research.google.com/github/akshkanwarcs24-gif/AI-LAB/blob/main/Unification6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:


import re

class Term:
    pass

class Variable(Term):
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return self.name

    def __eq__(self, other):
        return isinstance(other, Variable) and self.name == other.name

    def __hash__(self):
        return hash(self.name)

class Constant(Term):
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return self.name

    def __eq__(self, other):
        return isinstance(other, Constant) and self.name == other.name

    def __hash__(self):
        return hash(self.name)

class Compound(Term):
    def __init__(self, op, args):
        self.op = op
        self.args = args

    def __repr__(self):
        if not self.args:
            return self.op
        return f"{self.op}({', '.join(map(str, self.args))})"

    def __eq__(self, other):
        return isinstance(other, Compound) and self.op == other.op and self.args == other.args

    def __hash__(self):
        return hash((self.op, tuple(self.args)))

def parse_expression(text):
    text = text.replace(' ', '')
    tokens = re.findall(r'[a-zA-Z0-9_]+|\(|\)|,', text)
    index = 0

    def parse_term():
        nonlocal index
        token = tokens[index]
        index += 1
        if index < len(tokens) and tokens[index] == '(':
            index += 1
            args = []
            while tokens[index] != ')':
                args.append(parse_term())
                if tokens[index] == ',':
                    index += 1
            index += 1
            return Compound(token, args)
        else:

            if token[0].islower() and len(token) == 1 or token in ['x', 'y', 'z', 'u', 'v', 'w']:
                return Variable(token)
            else:
                return Constant(token)

    return parse_term()

def apply_substitution(expr, substitution):
    if isinstance(expr, Variable):
        if expr in substitution:
            return apply_substitution(substitution[expr], substitution)
        return expr
    elif isinstance(expr, Constant):
        return expr
    elif isinstance(expr, Compound):
        return Compound(expr.op, [apply_substitution(arg, substitution) for arg in expr.args])
    return expr

def occurs_in(var, expr, substitution):
    expr = apply_substitution(expr, substitution)
    if var == expr:
        return True
    elif isinstance(expr, Compound):
        return any(occurs_in(var, arg, substitution) for arg in expr.args)
    return False

def unify(E1, E2, substitution=None):
    if substitution is None:
        substitution = {}

    E1 = apply_substitution(E1, substitution)
    E2 = apply_substitution(E2, substitution)

    if E1 == E2:
        return substitution

    if isinstance(E1, Variable):
        return unify_variable(E1, E2, substitution)

    if isinstance(E2, Variable):
        return unify_variable(E2, E1, substitution)

    if isinstance(E1, Compound) and isinstance(E2, Compound):
        if E1.op == E2.op and len(E1.args) == len(E2.args):
            for a, b in zip(E1.args, E2.args):
                substitution = unify(a, b, substitution)
                if substitution is False:
                    return False
            return substitution

    return False

def unify_variable(v, e, substitution):
    if v == e:
        return substitution

    if v in substitution:
        return unify(substitution[v], e, substitution)



    if occurs_in(v, e, substitution):
        return False

    new_sub = dict(substitution)
    new_sub[v] = e
    return new_sub


try:
    print("=== Try your own expressions ===")
    user_e1 = input("Enter Expression 1 (e.g., Knows(john, x)): ")
    user_e2 = input("Enter Expression 2 (e.g., Knows(y, mother(y))): ")
    if user_e1 and user_e2:
        parsed_e1 = parse_expression(user_e1)
        parsed_e2 = parse_expression(user_e2)
        res = unify(parsed_e1, parsed_e2)
        print(f"\nParsed 1: {parsed_e1}")
        print(f"Parsed 2: {parsed_e2}")
        if res is False:
            print("Unification Result: FAILURE")
        else:

            formatted_subs = [f"{var}/{val}" for var, val in res.items()]
            print(f"Unification Result: {{{', '.join(formatted_subs)}}}")
except Exception as e:
    print(f"An error occurred or input was canceled: {e}")

=== Try your own expressions ===
Enter Expression 1 (e.g., Knows(john, x)): P(aksh,x)
Enter Expression 2 (e.g., Knows(y, mother(y))): P(y,aditya)

Parsed 1: P(aksh, x)
Parsed 2: P(y, aditya)
Unification Result: {y/aksh, x/aditya}
